# Week 11 — E: compressor reader

SAFE BY DEFAULT. Run All loads saved real results if present; otherwise it uses fabricated smoke fixtures (H/I read saved results only). No datasets/models are downloaded automatically. QASPER TEST is blocked.

For a user full run: set `RUN_FULL=True` and `RUN_SMOKE=False` in the next cell. Every later setup is derived automatically. Existing validated rows are reused. `OVERWRITE=True` is deliberately unsupported.



In [ ]:
RUN_SMOKE = True
RUN_FULL = False
RESUME = True
OVERWRITE = False
RUN_E_SMOKE = RUN_SMOKE
RUN_E_FULL = RUN_FULL
SMOKE_LIMIT = 3
REVIEW_PHASE = None
REVIEWED_CONFIG = None
REVIEW_REASON = None


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd().resolve()
while not (ROOT/'src/week11').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Open inside CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.week11.protocol import load, review
from src.week11.runner import execute, notebook_smoke
from src.week11.dashboard import status, resources
cfg = load()
pd.set_option('display.max_columns', 24)
pd.set_option('display.max_colwidth', 70)
configuration_df = pd.DataFrame([{'setting': k, 'value': str(v)} for k,v in cfg.items()])
display(configuration_df)
run_status_df = status(cfg)
display(run_status_df)
display(resources(cfg))


## Execute or resume

Synthetic output is stored only under `smoke/`. Pending token-compressor conditions are reported explicitly, never filled with invented scores. Full dependent phases require the previous manual freeze.

In [ ]:
if OVERWRITE: raise ValueError('Immutable results: use a new output version instead of overwrite')
if not RESUME: raise ValueError('Resume stays enabled; new experiments require a new configured version')
effective_smoke = notebook_smoke(cfg, 'E', RUN_SMOKE, RUN_FULL)
frames, figures = execute('E', cfg, full=RUN_FULL, explicit=RUN_FULL, smoke=effective_smoke, limit=SMOKE_LIMIT)
print('SYNTHETIC SMOKE ONLY' if effective_smoke else 'Saved/full result tables; inspect status before review')
display(frames.get('condition_status_df', pd.DataFrame()))


### compressor_reader_summary_df

In [ ]:
compressor_reader_summary_df = frames.get('compressor_reader_summary_df', pd.DataFrame())
display(compressor_reader_summary_df)

### compressor_quality_df

In [ ]:
compressor_quality_df = frames.get('compressor_quality_df', pd.DataFrame())
display(compressor_quality_df)

### compressor_efficiency_df

In [ ]:
compressor_efficiency_df = frames.get('compressor_efficiency_df', pd.DataFrame())
display(compressor_efficiency_df)

### compressor_pairwise_df

In [ ]:
compressor_pairwise_df = frames.get('compressor_pairwise_df', pd.DataFrame())
display(compressor_pairwise_df)

### compressor_interaction_df

In [ ]:
compressor_interaction_df = frames.get('compressor_interaction_df', pd.DataFrame())
display(compressor_interaction_df)

## Figures and underlying tables

Each PNG/PDF has a source CSV. Different datasets are plotted separately; cross-dataset displays use ranks/counts.

In [ ]:
for figure in figures:
    print(figure['name'], figure['dataset'])
    display(pd.read_csv(figure['data']))
    display(Image(filename=figure['png']))
if not figures: print('No completed data for these plots yet; no fabricated values were substituted.')


## Manual review / freeze

After inspecting real completed results, edit the review fields in the first FLAGS cell. Set `REVIEW_PHASE` to this phase, `REVIEWED_CONFIG` to a dictionary, and give a rationale. No winning setting is chosen automatically. Smoke cannot be frozen. If optional conditions remain pending, list them explicitly in `REVIEWED_CONFIG["waive_pending"]`, or qualify them before running.

In [ ]:
if REVIEW_PHASE is not None:
    if RUN_SMOKE: raise ValueError('Cannot freeze fabricated smoke output')
    if REVIEW_PHASE != 'E': raise ValueError('Review this notebook phase only')
    display(review(cfg, REVIEW_PHASE, REVIEWED_CONFIG, REVIEW_REASON))
else:
    print('Review not requested. No downstream configuration was selected automatically.')
display(status(cfg))
